In [1]:
# -*- coding: utf-8 -*-

04_context_aware.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1A8IJnYinwzvtZXnzzD7aWmOoObcGL-dW

# Context-Aware Recommender System

In this notebook, we build a context-aware recommender using the MovieLens dataset.

Unlike traditional recommenders, context-aware recommenders consider additional information beyond users and items. In this case, we use temporal context derived from timestamps.

The model uses:
- movie genres as item features
- item-item cosine similarity
- temporal context (weekday vs weekend)
- contextual pre-filtering to generate context-aware recommendations

Contributors: Andrés Ramírez, Yago Moreno

## 0. Import the libraries (Setup)

In [2]:
# Import the  necessary libraries

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load the Data

In [3]:
try:
    ratings = pd.read_csv("ml-latest-small/ratings.csv")
    movies = pd.read_csv("ml-latest-small/movies.csv")
except FileNotFoundError:
    ratings = pd.read_csv("ratings.csv")
    movies = pd.read_csv("movies.csv")

## 2. Temporal Train-Test Split

We use a temporal split so that recommendations are generated only from past interactions.

In [4]:
ratings = ratings.sort_values("timestamp")

cutoff = int(len(ratings) * 0.8)
train = ratings.iloc[:cutoff].copy()
test = ratings.iloc[cutoff:].copy()

print("Train size:", len(train))
print("Test size:", len(test))

Train size: 80668
Test size: 20168


## 3. Extract Context from Timestamp

We convert timestamps into calendar information and define a simple context variable: weekday vs weekend.

In [5]:
train["datetime"] = pd.to_datetime(train["timestamp"], unit="s")
test["datetime"] = pd.to_datetime(test["timestamp"], unit="s")

train["dayofweek"] = train["datetime"].dt.dayofweek
test["dayofweek"] = test["datetime"].dt.dayofweek

# Original binary context: weekday vs weekend
train["context"] = np.where(train["dayofweek"] >= 5, "weekend", "weekday")
test["context"] = np.where(test["dayofweek"] >= 5, "weekend", "weekday")

# --- IMPROVED CONTEXT: time_of_day ---
# morning: 06-11, afternoon: 12-17, evening: 18-21, night: 22-05
def get_time_of_day(hour):
    if 6 <= hour < 12:
        return "morning"
    elif 12 <= hour < 18:
        return "afternoon"
    elif 18 <= hour < 22:
        return "evening"
    else:
        return "night"

train["hour"] = train["datetime"].dt.hour
test["hour"] = test["datetime"].dt.hour

train["time_of_day"] = train["hour"].apply(get_time_of_day)
test["time_of_day"] = test["hour"].apply(get_time_of_day)

# --- RICH CONTEXT: combination of weekday/weekend + time_of_day ---
# e.g. "weekend_evening", "weekday_night"
# This gives up to 8 segments, which is still interpretable
train["context_rich"] = train["context"] + "_" + train["time_of_day"]
test["context_rich"] = test["context"] + "_" + test["time_of_day"]

print("Context distribution (train):")
print(train["context_rich"].value_counts())
train[["timestamp", "datetime", "context", "time_of_day", "context_rich"]].head()

Context distribution (train):
context_rich
weekday_night        19093
weekday_afternoon    16165
weekday_evening      13644
weekday_morning       9290
weekend_night         8038
weekend_afternoon     5365
weekend_evening       5241
weekend_morning       3832
Name: count, dtype: int64


,timestamp,datetime,context,time_of_day,context_rich
66719,828124615,1996-03-29 18:36:55,weekday,evening,weekday_evening
66716,828124615,1996-03-29 18:36:55,weekday,evening,weekday_evening
66717,828124615,1996-03-29 18:36:55,weekday,evening,weekday_evening
66718,828124615,1996-03-29 18:36:55,weekday,evening,weekday_evening
66712,828124615,1996-03-29 18:36:55,weekday,evening,weekday_evening


## 4. Create Genre-Based Item Features

To keep the notebook simple and consistent with Notebook 3, we use movie genres as item features.

In [6]:
movies_features = movies.copy()
movies_features["genres"] = movies_features["genres"].str.replace("|", " ", regex=False)

genre_matrix = movies_features["genres"].str.get_dummies(sep=" ")
genre_matrix.index = movies_features["movieId"]

genre_matrix.head()

,(no,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,...,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,genres,listed)
movieId,,,,,,,,,,,,,,,,,,,,,
1,0,0,1,1,1,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 5. Item-Item Similarity

We compute cosine similarity between movies based on their genre vectors.

In [7]:
item_similarity = cosine_similarity(genre_matrix)

item_similarity = pd.DataFrame(
    item_similarity,
    index=genre_matrix.index,
    columns=genre_matrix.index
)

item_similarity.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.774597,0.316228,0.258199,0.447214,0.0,0.316228,0.632456,0.0,0.258199,...,0.447214,0.316228,0.316228,0.447214,0.0,0.670820,0.774597,0.00000,0.316228,0.447214
2,0.774597,1.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.816497,0.0,0.333333,...,0.000000,0.000000,0.000000,0.000000,0.0,0.288675,0.333333,0.00000,0.000000,0.000000
3,0.316228,0.000000,1.000000,0.816497,0.707107,0.0,1.000000,0.000000,0.0,0.000000,...,0.353553,0.000000,0.500000,0.000000,0.0,0.353553,0.408248,0.00000,0.000000,0.707107
4,0.258199,0.000000,0.816497,1.000000,0.577350,0.0,0.816497,0.000000,0.0,0.000000,...,0.288675,0.408248,0.816497,0.000000,0.0,0.288675,0.333333,0.57735,0.000000,0.577350
5,0.447214,0.000000,0.707107,0.577350,1.000000,0.0,0.707107,0.000000,0.0,0.000000,...,0.500000,0.000000,0.707107,0.000000,0.0,0.500000,0.577350,0.00000,0.000000,1.000000


## 6. Recommend Movies for a User Under a Given Context

We use only the user's interactions that match the same context as the target situation.

In [8]:
def recommend_for_user_context(user_id, context_value, train_ratings, item_similarity, movies,
                               top_n=10, min_rating=4.0, context_col="context_rich"):
    """
    Generate context-aware recommendations for a user.
    context_col: column to use for filtering — 'context' (weekday/weekend) or
                 'context_rich' (weekday/weekend + time_of_day)
    """
    contextual_train = train_ratings[train_ratings[context_col] == context_value].copy()

    user_ratings = contextual_train[contextual_train["userId"] == user_id].copy()
    liked_movies = user_ratings[user_ratings["rating"] >= min_rating]["movieId"]

    if liked_movies.empty:
        return pd.DataFrame(columns=["movieId", "title", "score"])

    scores = pd.Series(dtype=float)

    for movie_id in liked_movies:
        if movie_id in item_similarity.index:
            similar_scores = item_similarity.loc[movie_id]
            scores = scores.add(similar_scores, fill_value=0)

    scores = scores.sort_values(ascending=False)

    seen_movies = user_ratings["movieId"].unique()
    scores = scores[~scores.index.isin(seen_movies)]

    recommendations = pd.DataFrame({
        "movieId": scores.index,
        "score": scores.values
    })

    recommendations = recommendations.merge(movies[["movieId", "title"]], on="movieId")

    return recommendations.head(top_n)[["movieId", "title", "score"]]

## 7. Example: Context-Aware Recommendations

In [9]:
example_user = train["userId"].iloc[0]
example_context = "weekend_evening"

print(f"Recommendations for user {example_user} in context: {example_context}")
recommend_for_user_context(example_user, example_context, train, item_similarity, movies,
                           top_n=10, min_rating=4.0, context_col="context_rich")

Recommendations for user 429 in context: weekend_evening


,movieId,title,score


## 8. Prediction Function for Notebook 5

For each user-item pair in the test set, we generate:

- `score_context`: a ranking score
- `predicted_rating_context`: a predicted rating for RMSE and MAE

The model uses only training interactions that match the same context as the target test instance.

In [10]:
def predict_context_metrics(user_id, movie_id, context_value, train_data, item_similarity,
                            min_rating=4.0, context_col="context_rich"):
    """
    Predict ranking score and rating for a user-item pair given a context.
    context_col: 'context' or 'context_rich'
    Falls back to full user history if no interactions found for the given context.
    """
    contextual_train = train_data[train_data[context_col] == context_value].copy()
    user_ratings = contextual_train[contextual_train["userId"] == user_id].copy()

    # Fallback: if no history in this specific context, use full user history
    if user_ratings.empty:
        user_ratings = train_data[train_data["userId"] == user_id].copy()

    liked_movies = user_ratings[user_ratings["rating"] >= min_rating][["movieId", "rating"]]

    if liked_movies.empty:
        return np.nan, np.nan

    if movie_id not in item_similarity.index:
        return np.nan, np.nan

    similarities = []
    ratings_list = []

    for _, row in liked_movies.iterrows():
        liked_movie = row["movieId"]
        liked_rating = row["rating"]

        if liked_movie in item_similarity.columns:
            sim = item_similarity.loc[movie_id, liked_movie]
            similarities.append(sim)
            ratings_list.append(liked_rating)

    if len(similarities) == 0:
        return np.nan, np.nan

    similarities = np.array(similarities)
    ratings_list = np.array(ratings_list)

    # Ranking score
    score_context = similarities.mean()

    # Predicted rating
    if similarities.sum() > 0:
        predicted_rating_context = np.dot(similarities, ratings_list) / similarities.sum()
    else:
        predicted_rating_context = ratings_list.mean()

    return score_context, predicted_rating_context

## 9. Compute Predictions on the Full Test Set

In [11]:
pred_context = test[["userId", "movieId", "rating", "context", "context_rich"]].copy()

pred_context[["score_context", "predicted_rating_context"]] = pred_context.apply(
    lambda row: pd.Series(
        predict_context_metrics(
            row["userId"],
            row["movieId"],
            row["context_rich"],          # use rich context
            train,
            item_similarity,
            min_rating=4.0,
            context_col="context_rich"    # match column name
        )
    ),
    axis=1
)

global_mean = train["rating"].mean()

pred_context["score_context"] = pred_context["score_context"].fillna(0)
pred_context["predicted_rating_context"] = pred_context["predicted_rating_context"].fillna(global_mean)

pred_context.head()

,userId,movieId,rating,context,context_rich,score_context,predicted_rating_context
79691,495,72998,5.0,weekday,weekday_morning,0.260112,4.783460
79564,495,2762,4.5,weekday,weekday_morning,0.340262,4.660192
79601,495,4993,0.5,weekday,weekday_morning,0.108660,4.825495
79709,495,89492,5.0,weekday,weekday_morning,0.487505,4.680301
79551,495,2028,4.5,weekday,weekday_morning,0.434230,4.641142


## 11. Interpretation

This recommender extends the content-based approach by incorporating temporal context.

Instead of using the full training history of a user, it only uses the interactions that match the same context as the target recommendation, in this case weekday or weekend.

This makes the recommender more sensitive to situational behavior. However, contextual pre-filtering also reduces the amount of available training data, which can increase sparsity.

## 12. Conclusion

In this notebook, we implemented a context-aware recommender using temporal context derived from timestamps.

First, we extracted weekday and weekend information from the interaction history. Then, we applied contextual pre-filtering so that recommendations were based only on interactions observed under the same context. Finally, we generated both a ranking score (`score_context`) and a predicted rating (`predicted_rating_context`) for each user-item pair in the test set.

This allows the context-aware model to be compared directly with the other recommenders in Notebook 5 using both ranking metrics and error-based metrics.

Overall, the context-aware approach adds situational information to the recommendation process, but its effectiveness depends on whether the additional context improves relevance more than it increases sparsity.

## 13. Export Predictions for Notebook 5

In [12]:
pred_context.to_csv("predictions_context.csv", index=False)

print("File exported: predictions_context.csv")
pred_context.head()

File exported: predictions_context.csv


,userId,movieId,rating,context,context_rich,score_context,predicted_rating_context
79691,495,72998,5.0,weekday,weekday_morning,0.260112,4.783460
79564,495,2762,4.5,weekday,weekday_morning,0.340262,4.660192
79601,495,4993,0.5,weekday,weekday_morning,0.108660,4.825495
79709,495,89492,5.0,weekday,weekday_morning,0.487505,4.680301
79551,495,2028,4.5,weekday,weekday_morning,0.434230,4.641142
